# QUEST 9
__Plotly__

In [51]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

In [52]:
database_path = "../data/checking-logs.sqlite"

In [53]:
try:
    connection = sqlite3.connect(database=database_path)
except Exception:
    print("The connection is failed :(")

## Создание таблицы

In [54]:
query = """
select timestamp, numTrials, uid from checker
where status = 'ready' and uid like 'user_%' and labname = 'project1'
"""
df = pd.io.sql.read_sql(query, connection, parse_dates="timestamp")
df["timestamp"] = df["timestamp"].dt.date

In [55]:
display(df)

,timestamp,numTrials,uid
0,2020-04-17,1,user_4
1,2020-04-17,2,user_4
2,2020-04-17,3,user_4
3,2020-04-17,4,user_4
4,2020-04-17,5,user_4
...,...,...,...
946,2020-05-15,26,user_19
947,2020-05-15,27,user_19
948,2020-05-15,28,user_19
949,2020-05-15,27,user_28


In [56]:
display(df[df["uid"] == "user_18"])

,timestamp,numTrials,uid
296,2020-05-11,1,user_18
301,2020-05-11,2,user_18
302,2020-05-11,3,user_18
303,2020-05-11,4,user_18
304,2020-05-11,5,user_18
306,2020-05-11,6,user_18
307,2020-05-11,7,user_18
595,2020-05-13,8,user_18
771,2020-05-14,9,user_18
781,2020-05-14,10,user_18


In [57]:
df = df.groupby(["timestamp", "uid"])["numTrials"].max().unstack()
df.index = np.arange(len(df.index))

In [58]:

df.iloc[0] = df.iloc[0].fillna(0, axis=0)
df.fillna(method="pad", axis=0, inplace=True)
#df = df.transpose()

In [59]:
display(df)

uid,user_1,user_10,user_11,user_12,user_13,user_14,user_15,user_16,user_17,user_18,...,user_26,user_27,user_28,user_29,user_3,user_30,user_31,user_4,user_6,user_8
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,11.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,20.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,27.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.0,35.0,0.0,0.0
7,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,48.0,0.0,0.0
8,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,53.0,0.0,0.0
9,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,53.0,0.0,0.0


## Создание графика

Создадим 27 линий графика для динамики каждого пользователя:

In [60]:
initial_data = [go.Scatter(x=np.array([]), y =np.array([1]), mode = "lines+markers", name = uid) for uid in df.columns]

Создадим кадры для анимации для каждого пользователя - матрицу из значений функций в каждом x:

In [61]:
framess = []
for x in range(len(df.index)):
    frame = []
    for y in range(len(df.columns)):
        frame.append(go.Scatter(x=np.arange(x+1),
                                # y=np.array(df.loc[df.index < x, df.columns[y]]), # вывести значения колонки с индексом y где индекс таблицы меньше x
                                y = np.array(df.iloc[:(x+1), y]),
                                mode="lines+markers",
                                name=df.columns[y]
                                ))
    framme = {"data": frame}
    framess.append(framme)



In [62]:
fig = go.Figure(
    data=initial_data,
    layout={
        'width': 1100,
        'height': 500,
        'xaxis': {'range': (0,21)},
        'yaxis': {'range': (0, 160)},
        'title': 'Dynamic of commits per user in project1',
        'updatemenus': [{
            'type': 'buttons',
            'buttons': [{'method': 'animate', 'label': 'play', 'args': [None]}]
            }]}, frames=[{"data": f["data"]} for f in framess])     # список словарей с key = "data"
fig.show()

In [63]:
connection.close()